In [ ]:
# =========================================
# IMPORTS Y CONFIGURACIÓN
# =========================================

import os
import smtplib

from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from openai import OpenAI

from langchain_openai import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, AgentType
from langchain.tools import Tool


# =========================================
# CLIENTE OPENAI
# =========================================

client = OpenAI(
    api_key="GITHUB_API_KEY",
    base_url="GITHUB_BASE_URL"
)


# =========================================
# LLM CON LANGCHAIN
# =========================================

llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.7,
    api_key="GITHUB_API_KEY",
    base_url="GITHUB_BASE_URL"
)


# =========================================
# MEMORIA CORTO PLAZO
# =========================================

memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)


# =========================================
# MEMORIA LARGO PLAZO LOCAL
# =========================================

historial_global = []

solicitudes_frecuentes = {
    "permiso": 0,
    "multas": 0,
    "patentes": 0,
    "servicios": 0
}

# NUEVO
correos_suscritos = []


# =========================================
# TOOLS DE AGENTES
# =========================================

def tool_permiso(input_text):

    solicitudes_frecuentes["permiso"] += 1

    return """
    Para renovar el permiso de circulación necesitas:

    - SOAP vigente
    - Revisión técnica al día
    - Permiso anterior
    - Padrón del vehículo

    Puedes realizar el trámite en la Municipalidad
    de Llanquihue.
    """


def tool_multas(input_text):

    solicitudes_frecuentes["multas"] += 1

    return """
    El pago de multas puede realizarse:

    - Online
    - Presencialmente en oficinas municipales

    Debes presentar la información del vehículo
    o número de infracción.
    """


def tool_patentes(input_text):

    solicitudes_frecuentes["patentes"] += 1

    return """
    Para obtener una patente comercial necesitas:

    - Inicio de actividades
    - RUT empresa o persona
    - Dirección comercial
    - Permisos sanitarios si corresponde
    """


def tool_servicios(input_text):

    solicitudes_frecuentes["servicios"] += 1

    return """
    La Municipalidad de Llanquihue ofrece:

    - Aseo y ornato
    - Reciclaje
    - Pago de permisos
    - Atención social
    - Información comunitaria
    """


# =========================================
# REGISTRO DE TOOLS
# =========================================

tools = [

    Tool(
        name="Permisos",
        func=tool_permiso,
        description="Usa esta herramienta para consultas sobre permisos de circulación"
    ),

    Tool(
        name="Multas",
        func=tool_multas,
        description="Usa esta herramienta para consultas sobre multas de tránsito"
    ),

    Tool(
        name="Patentes",
        func=tool_patentes,
        description="Usa esta herramienta para consultas sobre patentes comerciales"
    ),

    Tool(
        name="Servicios",
        func=tool_servicios,
        description="Usa esta herramienta para consultas sobre servicios municipales"
    )
]


# =========================================
# CREACIÓN DEL AGENTE
# =========================================

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True
)


# =========================================
# FUNCIÓN GENERAL IA
# =========================================

def consultar_muni(pregunta, contexto, tipo="zero-shot"):

    try:

        # ZERO-SHOT
        if tipo == "zero-shot":

            messages = [
                {
                    "role": "system",
                    "content": contexto +
                    ". Responde de forma clara, breve y útil para ciudadanos de Chile."
                },

                {
                    "role": "user",
                    "content": pregunta
                }
            ]

        # FEW-SHOT
        elif tipo == "few-shot":

            messages = [
                {
                    "role": "system",
                    "content": contexto +
                    ". Responde como asistente municipal claro y práctico."
                },

                {
                    "role": "user",
                    "content": pregunta
                }
            ]

        # CHAIN OF THOUGHT
        elif tipo == "chain-of-thought":

            messages = [
                {
                    "role": "system",
                    "content": contexto +
                    ". Responde siguiendo este formato:\n"
                    "1. Explicación breve\n"
                    "2. Pasos a seguir\n"
                    "3. Recomendación final"
                },

                {
                    "role": "user",
                    "content": pregunta
                }
            ]

        else:

            messages = [
                {
                    "role": "system",
                    "content": contexto
                },

                {
                    "role": "user",
                    "content": pregunta
                }
            ]

        response = client.chat.completions.create(
            model="gpt-4o",
            messages=messages,
            temperature=0.7,
            max_tokens=300
        )

        print("\nRespuesta:")
        print(response.choices[0].message.content, "\n")

    except Exception as e:
        print("Error:", e)


# =========================================
# ELEGIR TIPO DE RESPUESTA
# =========================================

def elegir_tipo_respuesta():

    print("\nTipo de respuesta:")
    print("1. Zero-shot")
    print("2. Few-shot")
    print("3. Chain of Thought")

    tipo_op = input("Elige tipo: ")

    tipo_map = {
        "1": "zero-shot",
        "2": "few-shot",
        "3": "chain-of-thought"
    }

    return tipo_map.get(tipo_op, "zero-shot")


# =========================================
# FUNCIONES MUNICIPALES
# =========================================

def permiso_circulacion():

    print("\nPERMISO DE CIRCULACIÓN")

    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()

    contexto = "Eres un asistente de la Municipalidad de Llanquihue experto en permisos de circulación en Chile"

    consultar_muni(pregunta, contexto, tipo)


def patentes_comerciales():

    print("\nPATENTES COMERCIALES")

    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()

    contexto = "Eres un asistente experto en patentes comerciales municipales en Chile"

    consultar_muni(pregunta, contexto, tipo)


def pago_multas():

    print("\nPAGO DE MULTAS")

    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()

    contexto = "Eres un asistente que ayuda con el pago de multas de tránsito en municipalidades chilenas"

    consultar_muni(pregunta, contexto, tipo)


def servicios_varios():

    print("\nSERVICIOS VARIOS")

    pregunta = input("¿Qué deseas saber?: ")

    tipo = elegir_tipo_respuesta()

    contexto = "Eres un asistente municipal que informa sobre servicios generales de la Municipalidad de Llanquihue"

    consultar_muni(pregunta, contexto, tipo)


# =========================================
# RECIBIR NOTICIAS MUNICIPALES
# =========================================

def recibir_noticias():

    print("\nRECIBIR NOTICIAS DE LLANQUIHUE")

    correo = input("Ingresa tu correo electrónico: ")

    if correo not in correos_suscritos:
        correos_suscritos.append(correo)

    print("Correo registrado correctamente.")

    # =====================================
    # CONFIGURACIÓN DEL CORREO
    # =====================================

    remitente = "botmuni46@gmail.com"

    # CONTRASEÑA DE APLICACIÓN DE GOOGLE
    contraseña = "wjus ipwj dspa zzyw"

    asunto = "Noticias Municipalidad de Llanquihue"

    mensaje_texto = """
    Hola vecino/a de Llanquihue.

    El camión de basura pasará de lunes a sábado
    desde las 09:00 hasta las 18:00 horas.

    Municipalidad de Llanquihue.
    """

    try:

        mensaje = MIMEMultipart()

        mensaje["From"] = remitente
        mensaje["To"] = correo
        mensaje["Subject"] = asunto

        mensaje.attach(
            MIMEText(mensaje_texto, "plain")
        )

        servidor = smtplib.SMTP(
            "smtp.gmail.com",
            587
        )

        servidor.starttls()

        servidor.login(
            remitente,
            contraseña
        )

        servidor.sendmail(
            remitente,
            correo,
            mensaje.as_string()
        )

        servidor.quit()

        print("Noticia enviada correctamente.")

    except Exception as e:
        print("Error al enviar correo:", e)


# =========================================
# CHAT GENERAL INTELIGENTE
# =========================================

def chat_general():

    print("\nChat inteligente municipal (escribe 'salir')\n")

    tipo = elegir_tipo_respuesta()

    while True:

        user_input = input("Tú: ")

        if user_input.lower() == "salir":
            break

        # MEMORIA LARGO PLAZO
        historial_global.append(user_input)

        try:

            contexto_chat = f"""

            Eres un asistente inteligente
            de la Municipalidad de Llanquihue.

            Historial reciente:
            {historial_global[-5:]}

            Solicitudes frecuentes:
            {solicitudes_frecuentes}

            Tipo de respuesta:
            {tipo}

            """

            respuesta = agent.run(
                contexto_chat + "\nUsuario: " + user_input
            )

            print("\nIA:", respuesta, "\n")

        except Exception as e:
            print("Error:", e)


# =========================================
# MENÚ MUNICIPAL
# =========================================

def menu():

    while True:

        print("\n====== MUNICIPALIDAD DE LLANQUIHUE ======")
        print("1. Permiso de Circulación")
        print("2. Patentes Comerciales")
        print("3. Pago de Multas")
        print("4. Servicios Varios")
        print("5. Chat General Inteligente")
        print("6. Recibir Noticias de Llanquihue")
        print("7. Ver Solicitudes Frecuentes")
        print("8. Ver Historial")
        print("9. Salir")

        opcion = input("Selecciona una opción: ")

        if opcion == "1":

            permiso_circulacion()

        elif opcion == "2":

            patentes_comerciales()

        elif opcion == "3":

            pago_multas()

        elif opcion == "4":

            servicios_varios()

        elif opcion == "5":

            chat_general()

        elif opcion == "6":

            recibir_noticias()

        elif opcion == "7":

            print("\nSOLICITUDES FRECUENTES")
            print(solicitudes_frecuentes)

        elif opcion == "8":

            print("\nHISTORIAL RECIENTE")

            for item in historial_global[-10:]:
                print("-", item)

        elif opcion == "9":

            print("Saliendo...")
            break

        else:

            print("Opción inválida")


# =========================================
# EJECUCIÓN
# =========================================

if __name__ == "__main__":
    menu()

C:\Users\W608-PCXX\AppData\Local\Temp\ipykernel_16816\1169683563.py:126: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 1.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  agent = initialize_agent(



====== MUNICIPALIDAD DE LLANQUIHUE ======
1. Permiso de Circulación
2. Patentes Comerciales
3. Pago de Multas
4. Servicios Varios
5. Chat General Inteligente
6. Ver Solicitudes Frecuentes
7. Ver Historial
8. Salir


Selecciona una opción:  8


Saliendo...
